# Interpretable Credit Scoring with Explainable Boosting Machines (EBM)

**Project 1 of 3** — Credit risk → Model monitoring → Financial crime (PU learning)

## Business problem
A bank must decide **who to lend to**. The model must be:
1. **Accurate** — fewer bad loans, more good customers approved.
2. **Explainable** — regulators (e.g. MAS FEAT principles), model risk teams and customers need to know *why* a loan was rejected.

Black-box models (XGBoost) are accurate but hard to explain. Logistic regression is easy to explain but less accurate.
**EBM** (a glass-box Generalized Additive Model with boosting) tries to give both.

## Questions this notebook answers
- How close is EBM to XGBoost in accuracy?
- What drives default risk, and does it make business sense?
- Can we give each applicant clear **reason codes** ("why was I rejected?")?
- What is the business impact of using the model for approval decisions?

## Data
[Lending Club loan data](https://www.kaggle.com/datasets/wordsforthewise/lending-club) (2007–2018Q4, `accepted_2007_to_2018Q4.csv.gz`).

```bash
pip install interpret xgboost scikit-learn pandas matplotlib joblib
# if `interpret` fails to install, use: pip install interpret-core
kaggle datasets download -d wordsforthewise/lending-club -p data/
```

## 1. Setup

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier
from interpret.glassbox import ExplainableBoostingClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

# Plot style: one fixed color per model, used in every chart
COLORS = {"EBM": "#2a78d6", "XGBoost": "#eb6834", "LogReg": "#1baf7a"}
INTERACTION = "#6250d6"
RISK_UP, RISK_DOWN = "#e34948", "#008300"
GREY = "#8a8985"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "axes.edgecolor": "#b5b4ae", "axes.axisbelow": True, "axes.titleweight": "bold", "lines.linewidth": 2,
})

### Config
- **Out-of-time split**: train on old loans, test on newer loans. This is how a bank really uses a model.
- Only **36-month loans issued 2010–2015**, so every loan has finished (matured) by end of 2018 and we know the true outcome.
- `USE_LC_GRADE = False`: Lending Club's own `grade` / `int_rate` are the output of *their* risk model. Using them means we just copy their model. We build our own score from applicant data only.

In [ ]:
DATA_PATH = os.environ.get("LC_DATA_PATH", "data/accepted_2007_to_2018Q4.csv.gz")
ARTIFACT_DIR = "artifacts"

TRAIN_YEARS = [2010, 2011, 2012, 2013]
VALID_YEAR = 2014          # early stopping / model selection
TEST_YEAR = 2015           # final, untouched out-of-time test
SAMPLE_N = 200_000         # cap training rows (EBM is slower than XGBoost). None = use all
USE_LC_GRADE = False
RANDOM_STATE = 42

os.makedirs(ARTIFACT_DIR, exist_ok=True)

## 2. Load data
We load **only features known at application time**. Columns like `total_pymnt`, `recoveries`, `last_fico_range_high` are known only *after* the loan starts → using them is **target leakage**.

In [ ]:
RAW_COLS = [
    # application-time information
    "loan_amnt", "term", "int_rate", "installment", "grade", "sub_grade", "emp_length",
    "home_ownership", "annual_inc", "verification_status", "issue_d", "purpose", "dti",
    "delinq_2yrs", "earliest_cr_line", "fico_range_low", "fico_range_high", "inq_last_6mths",
    "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc", "mort_acc",
    "pub_rec_bankruptcies", "application_type",
    # outcome
    "loan_status",
]

t0 = time.time()
raw = pd.read_csv(DATA_PATH, usecols=RAW_COLS, low_memory=False)
print(f"Loaded {raw.shape[0]:,} rows x {raw.shape[1]} cols in {time.time() - t0:.0f}s")
raw.head(3)

## 3. Target definition
- `default = 1`: Charged Off / Default
- `default = 0`: Fully Paid
- Loans still `Current` / `Late` are dropped (outcome unknown).

In [ ]:
df = raw.copy()
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
df = df.dropna(subset=["issue_d"])
df["issue_year"] = df["issue_d"].dt.year

status = df["loan_status"].astype(str).str.replace("Does not meet the credit policy. Status:", "", regex=False)
df["default"] = status.map({"Fully Paid": 0, "Charged Off": 1, "Default": 1})
df["term_months"] = pd.to_numeric(df["term"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")

keep = (
    df["default"].notna()
    & (df["term_months"] == 36)
    & df["issue_year"].between(min(TRAIN_YEARS), TEST_YEAR)
)
df = df.loc[keep].copy()
df["default"] = df["default"].astype(int)

print(f"Modeling rows: {len(df):,}   overall default rate: {df['default'].mean():.1%}")
df.groupby("issue_year")["default"].agg(n_loans="size", default_rate="mean")

## 4. Feature engineering

In [ ]:
def to_num(s):
    # handles both "13.5%" strings and floats
    return pd.to_numeric(s.astype(str).str.replace("%", "", regex=False).str.strip(), errors="coerce")

df["int_rate"] = to_num(df["int_rate"])
df["revol_util"] = to_num(df["revol_util"])

emp = df["emp_length"].astype(str).str.replace("< 1", "0", regex=False).str.extract(r"(\d+)")[0]
df["emp_length_yrs"] = pd.to_numeric(emp, errors="coerce")

ecl = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y", errors="coerce")
df["credit_hist_months"] = ((df["issue_d"] - ecl).dt.days / 30.44).round()

df["fico"] = (df["fico_range_low"] + df["fico_range_high"]) / 2
df["annual_inc"] = df["annual_inc"].replace(0, np.nan)
df["loan_to_income"] = df["loan_amnt"] / df["annual_inc"]
df["installment_to_income"] = df["installment"] * 12 / df["annual_inc"]

df["home_ownership"] = df["home_ownership"].replace({"ANY": "OTHER", "NONE": "OTHER"})

NUM_FEATURES = [
    "loan_amnt", "annual_inc", "loan_to_income", "dti", "fico", "emp_length_yrs",
    "credit_hist_months", "delinq_2yrs", "inq_last_6mths", "open_acc", "pub_rec",
    "revol_bal", "revol_util", "total_acc", "mort_acc", "pub_rec_bankruptcies",
]
CAT_FEATURES = ["home_ownership", "verification_status", "purpose", "application_type"]

if USE_LC_GRADE:
    # installment is calculated from int_rate, so it also carries LC's pricing
    NUM_FEATURES += ["int_rate", "installment_to_income"]
    CAT_FEATURES += ["sub_grade"]

FEATURES = NUM_FEATURES + CAT_FEATURES
for c in CAT_FEATURES:
    df[c] = df[c].fillna("Missing").astype(str)

print(len(FEATURES), "features")
df[FEATURES].describe().T[["count", "mean", "50%", "min", "max"]]

## 5. Quick EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

q = df.groupby(df["issue_d"].dt.to_period("Q"))["default"].mean() * 100
axes[0].plot(range(len(q)), q.values, color=COLORS["EBM"], marker="o", markersize=4)
axes[0].set_xticks(range(0, len(q), 4), [str(p) for p in q.index[::4]])
axes[0].set_title("Default rate by issue quarter")
axes[0].set_ylabel("Default rate (%)")
for y, name in [(TRAIN_YEARS[-1], "train | valid"), (VALID_YEAR, "valid | test")]:
    pos = np.searchsorted([p.year for p in q.index], y, side="right") - 0.5
    axes[0].axvline(pos, color=GREY, ls="--", lw=1)
    axes[0].text(pos + 0.3, 0.03, name, transform=axes[0].get_xaxis_transform(), ha="left", va="bottom", fontsize=8, color=GREY)

bands = pd.cut(df["fico"], bins=[600, 660, 680, 700, 720, 740, 760, 850])
fb = df.groupby(bands, observed=True)["default"].mean() * 100
axes[1].bar(range(len(fb)), fb.values, color=COLORS["EBM"], width=0.7)
axes[1].set_xticks(range(len(fb)), [str(b) for b in fb.index], rotation=30)
axes[1].set_title("Default rate by FICO band")
axes[1].set_ylabel("Default rate (%)")
plt.tight_layout(); plt.show()

## 6. Out-of-time split

In [ ]:
train = df[df["issue_year"].isin(TRAIN_YEARS)]
valid = df[df["issue_year"] == VALID_YEAR]
test = df[df["issue_year"] == TEST_YEAR]
if SAMPLE_N and len(train) > SAMPLE_N:
    train = train.sample(SAMPLE_N, random_state=RANDOM_STATE)

X_tr, y_tr = train[FEATURES], train["default"]
X_va, y_va = valid[FEATURES], valid["default"]
X_te, y_te = test[FEATURES], test["default"]

pd.DataFrame({
    "rows": [len(train), len(valid), len(test)],
    "default_rate": [y_tr.mean(), y_va.mean(), y_te.mean()],
}, index=[f"train {TRAIN_YEARS[0]}-{TRAIN_YEARS[-1]}", f"valid {VALID_YEAR}", f"test {TEST_YEAR}"])

## 7. Models
| Model | Role |
|---|---|
| Logistic Regression | Traditional scorecard baseline — fully explainable |
| XGBoost | Black-box benchmark — usually most accurate |
| **EBM** | Glass-box — the candidate we want to use |

In [ ]:
# --- Logistic regression (scorecard-style baseline)
preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), NUM_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01), CAT_FEATURES),
])
logreg = make_pipeline(preprocess, LogisticRegression(max_iter=2000))
t0 = time.time(); logreg.fit(X_tr, y_tr)
print(f"LogReg   fit {time.time() - t0:5.0f}s")

# --- XGBoost (native categorical support)
CAT_LEVELS = {c: sorted(X_tr[c].unique()) for c in CAT_FEATURES}
def to_xgb(X):
    X = X.copy()
    for c in CAT_FEATURES:
        X[c] = pd.Categorical(X[c], categories=CAT_LEVELS[c])
    return X

xgb = XGBClassifier(
    n_estimators=2000, learning_rate=0.03, max_depth=4, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=50, enable_categorical=True, tree_method="hist", eval_metric="auc",
    early_stopping_rounds=100, random_state=RANDOM_STATE, n_jobs=-1,
)
t0 = time.time(); xgb.fit(to_xgb(X_tr), y_tr, eval_set=[(to_xgb(X_va), y_va)], verbose=False)
print(f"XGBoost  fit {time.time() - t0:5.0f}s  (best iteration {xgb.best_iteration})")

# --- EBM
ebm = ExplainableBoostingClassifier(
    interactions=10,            # allow up to 10 pairwise interactions
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
t0 = time.time(); ebm.fit(X_tr, y_tr)
print(f"EBM      fit {time.time() - t0:5.0f}s")

## 8. Performance
Credit risk teams usually look at **Gini** (= 2·AUC − 1) and **KS** (max separation between good and bad customers).

In [ ]:
def ks_stat(y, p):
    fpr, tpr, _ = roc_curve(y, p)
    return np.max(tpr - fpr)

def evaluate(y, p):
    auc = roc_auc_score(y, p)
    return {"AUC": auc, "Gini": 2 * auc - 1, "KS": ks_stat(y, p),
            "PR-AUC": average_precision_score(y, p), "Brier": brier_score_loss(y, p)}

preds = {}
for split, X, y in [("valid", X_va, y_va), ("test", X_te, y_te)]:
    preds[split] = {
        "EBM": ebm.predict_proba(X)[:, 1],
        "XGBoost": xgb.predict_proba(to_xgb(X))[:, 1],
        "LogReg": logreg.predict_proba(X)[:, 1],
    }

results = pd.DataFrame({
    (split, m): evaluate(y, p)
    for split, y in [("valid", y_va), ("test", y_te)]
    for m, p in preds[split].items()
}).T
results

In [ ]:
p_te = preds["test"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for m, p in p_te.items():
    fpr, tpr, _ = roc_curve(y_te, p)
    axes[0].plot(fpr, tpr, color=COLORS[m], label=f"{m}  (AUC {roc_auc_score(y_te, p):.3f})")
axes[0].plot([0, 1], [0, 1], color=GREY, ls="--", lw=1)
axes[0].set(title=f"ROC curve — test {TEST_YEAR}", xlabel="False positive rate", ylabel="True positive rate")
axes[0].legend(loc="lower right", frameon=False)

for m, p in p_te.items():
    frac_pos, mean_pred = calibration_curve(y_te, p, n_bins=10, strategy="quantile")
    axes[1].plot(mean_pred * 100, frac_pos * 100, color=COLORS[m], marker="o", markersize=5, label=m)
lim = max(axes[1].get_xlim()[1], axes[1].get_ylim()[1])
axes[1].plot([0, lim], [0, lim], color=GREY, ls="--", lw=1)
axes[1].set(title="Calibration — predicted vs actual default rate",
            xlabel="Predicted default rate (%)", ylabel="Actual default rate (%)")
axes[1].legend(loc="upper left", frameon=False)
plt.tight_layout(); plt.show()

> **Check**: calibration on the test year. If the dots are below the diagonal, the model over-estimates risk in the new year (default rate changed over time). This is a first sign of **drift** → see Project 2 (monitoring).

## 9. Global explanation — what drives default risk?
EBM is additive: `logit(p) = intercept + f1(fico) + f2(dti) + ... + f12(x1, x2)`.
Each `f` is a learned curve we can plot and check with business sense.

In [ ]:
imp = pd.Series(ebm.term_importances(), index=ebm.term_names_).sort_values()
top = imp.tail(15)

fig, ax = plt.subplots(figsize=(8, 5.5))
is_inter = [" & " in n for n in top.index]
ax.barh(top.index, top.values, color=[INTERACTION if i else COLORS["EBM"] for i in is_inter], height=0.65)
ax.set_title("EBM term importance (mean |contribution| in log-odds)")
ax.set_xlabel("Importance")
ax.grid(axis="y", visible=False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=COLORS["EBM"], label="Single feature"),
                   Patch(color=INTERACTION, label="Interaction")], frameon=False, loc="lower right")
plt.tight_layout(); plt.show()

In [ ]:
from matplotlib.ticker import FuncFormatter, MaxNLocator

def _short(x, _):
    return f"{x/1000:.0f}k" if abs(x) >= 1_000 else f"{x:g}"

def plot_shape(ebm, feature, ax):
    # Plot an EBM main-effect shape function with its uncertainty band
    i = list(ebm.term_names_).index(feature)
    d = ebm.explain_global().data(i)
    scores = np.asarray(d["scores"], dtype=float)
    lo = np.asarray(d.get("lower_bounds", scores), dtype=float)
    hi = np.asarray(d.get("upper_bounds", scores), dtype=float)
    names = d["names"]
    is_cont = ebm.feature_types_in_[list(ebm.feature_names_in_).index(feature)] == "continuous"
    if is_cont:
        edges = np.asarray(names, dtype=float)[:-1]
        # show the 1st-99th percentile of training data, where most customers are
        x_lo, x_hi = np.nanpercentile(X_tr[feature], [1, 99])
        vis = (edges >= x_lo) & (edges <= x_hi)
        ax.fill_between(edges, lo, hi, step="post", color=COLORS["EBM"], alpha=0.18, lw=0)
        ax.step(edges, scores, where="post", color=COLORS["EBM"])
        ax.set_xlim(x_lo, x_hi)
        if vis.any():
            pad = 0.1 * (hi[vis].max() - lo[vis].min() + 1e-9)
            ax.set_ylim(lo[vis].min() - pad, hi[vis].max() + pad)
        ax.axhline(0, color=GREY, lw=1)
        ax.xaxis.set_major_locator(MaxNLocator(5))
        ax.xaxis.set_major_formatter(FuncFormatter(_short))
    else:
        order = np.argsort(scores)
        ax.barh(np.array(names)[order], scores[order], color=COLORS["EBM"], height=0.6,
                xerr=[(scores - lo)[order], (hi - scores)[order]], error_kw={"ecolor": GREY, "lw": 1})
        ax.grid(axis="y", visible=False)
        ax.axvline(0, color=GREY, lw=1)
    ax.set_title(feature)

main_effects = [n for n in imp.sort_values(ascending=False).index if " & " not in n][:6]
fig, axes = plt.subplots(2, 3, figsize=(14, 7.5))
for f, ax in zip(main_effects, axes.ravel()):
    plot_shape(ebm, f, ax)
fig.suptitle("Top 6 shape functions (y = contribution to log-odds of default; higher = riskier)", y=1.0)
plt.tight_layout(); plt.show()

**Sanity check with business sense** (write your own observations here):
- Does risk go **down** when FICO goes **up**?
- Does risk go **up** with DTI, loan-to-income and recent credit inquiries?
- Any strange jumps? → could be data issues or overfitting. EBM lets us **see and fix** these (e.g. `ebm.monotonize("fico")` or editing the curve), which is impossible with a black-box model.

## 10. Local explanation — reason codes
For a rejected applicant, a bank must explain **why**. EBM gives the exact contribution of each feature — no approximation like SHAP on a black box.

In [ ]:
contrib = pd.DataFrame(ebm.eval_terms(X_te), columns=ebm.term_names_, index=X_te.index)

def reason_codes(idx, k=3):
    c = contrib.loc[idx].sort_values(ascending=False).head(k)
    rows = []
    for term, val in c.items():
        feats = term.split(" & ")
        value = ", ".join(f"{f}={X_te.at[idx, f]:.4g}" if isinstance(X_te.at[idx, f], (int, float, np.number))
                          else f"{f}={X_te.at[idx, f]}" for f in feats)
        rows.append({"reason": term, "applicant value": value, "log-odds contribution": round(val, 3)})
    return pd.DataFrame(rows)

# Example: the highest-risk applicant in the test set
p_ebm = p_te["EBM"]
riskiest = X_te.index[np.argmax(p_ebm)]
print(f"Predicted default probability: {p_ebm.max():.1%}   (actual default = {y_te.loc[riskiest]})")
reason_codes(riskiest)

In [ ]:
# Waterfall-style view for the same applicant
c = contrib.loc[riskiest]
top_terms = c.abs().sort_values(ascending=False).head(10).index
vals = c[top_terms].sort_values()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(vals.index, vals.values, color=[RISK_UP if v > 0 else RISK_DOWN for v in vals.values], height=0.65)
ax.axvline(0, color=GREY, lw=1)
ax.set_title("Why is this applicant high risk? (top 10 terms)")
ax.set_xlabel("Contribution to log-odds  (→ riskier)")
ax.grid(axis="y", visible=False)
ax.legend(handles=[Patch(color=RISK_UP, label="Increases risk"),
                   Patch(color=RISK_DOWN, label="Decreases risk")], frameon=False, loc="lower right")
plt.tight_layout(); plt.show()

## 11. Business impact — approval strategy
A score is useful only if it changes decisions. If the bank **rejects the riskiest X%** of applicants, how much does the bad rate of approved loans fall?

In [ ]:
def approval_curve(y, p, rates=np.linspace(0.5, 1.0, 51)):
    order = np.argsort(p)               # lowest risk first
    y_sorted = np.asarray(y)[order]
    out = []
    for r in rates:
        n = int(len(y_sorted) * r)
        out.append((r, y_sorted[:n].mean()))
    return pd.DataFrame(out, columns=["approval_rate", "bad_rate"])

fig, ax = plt.subplots(figsize=(8, 4.5))
for m, p in p_te.items():
    ac = approval_curve(y_te, p)
    ax.plot(ac["approval_rate"] * 100, ac["bad_rate"] * 100, color=COLORS[m], label=m)
ax.set(title=f"Approval rate vs bad rate of approved loans — test {TEST_YEAR}",
       xlabel="Approval rate (%)", ylabel="Bad rate among approved (%)")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

base = y_te.mean()
rows = []
for r in [0.95, 0.90, 0.80]:
    row = {"approval_rate": f"{r:.0%}"}
    for m, p in p_te.items():
        row[m] = approval_curve(y_te, p, [r])["bad_rate"].iloc[0]
    rows.append(row)
impact = pd.DataFrame(rows).set_index("approval_rate")
print(f"Bad rate if we approve everyone: {base:.2%}")
impact.style.format("{:.2%}")

In [ ]:
# Score bands (deciles) — how a credit team would read the score
band = pd.qcut(p_ebm, 10, labels=[f"D{i}" for i in range(1, 11)])   # D1 = lowest risk
decile = (pd.DataFrame({"band": band, "default": y_te.values, "pd": p_ebm})
          .groupby("band", observed=True)
          .agg(n=("default", "size"), predicted_pd=("pd", "mean"), actual_default=("default", "mean")))
decile["share_of_all_defaults"] = decile["n"] * decile["actual_default"] / y_te.sum()
decile

## 12. Save artifacts for Project 2 (model monitoring)

In [ ]:
joblib.dump({"ebm": ebm, "xgb": xgb, "logreg": logreg, "features": FEATURES,
             "num_features": NUM_FEATURES, "cat_features": CAT_FEATURES, "cat_levels": CAT_LEVELS,
             "train_years": TRAIN_YEARS, "valid_year": VALID_YEAR, "test_year": TEST_YEAR},
            f"{ARTIFACT_DIR}/credit_models.joblib")
df[FEATURES + ["issue_d", "issue_year", "default"]].to_pickle(f"{ARTIFACT_DIR}/modeling_data.pkl")
print("Saved:", os.listdir(ARTIFACT_DIR))

## 13. Summary for the interview
The cell below prints the key numbers. Use them to fill the story:

1. **Problem** — Banks need accurate *and* explainable credit models (regulation, customer trust).
2. **Approach** — Compared a scorecard-style logistic regression, XGBoost and EBM on an **out-of-time** test year, with application-time features only (no leakage, no LC grade).
3. **Result** — EBM is close to XGBoost in accuracy, but every prediction is fully explainable.
4. **Business value** — Clear reason codes for each applicant; shape functions can be reviewed by the risk team; approval-rate vs bad-rate trade-off for policy decisions.
5. **Next step** — The model's calibration changes in the new year → we need monitoring (**Project 2**).

In [ ]:
t = results.loc["test"]
gap = t.loc["XGBoost", "AUC"] - t.loc["EBM", "AUC"]
r90 = impact.loc["90%"]
print(f'''
Test year {TEST_YEAR} (out-of-time, {len(y_te):,} loans, default rate {base:.1%})
- AUC:  EBM {t.loc["EBM","AUC"]:.3f} | XGBoost {t.loc["XGBoost","AUC"]:.3f} | LogReg {t.loc["LogReg","AUC"]:.3f}
- EBM is {abs(gap)*100:.1f} AUC points {"below" if gap > 0 else "above"} XGBoost, and fully explainable.
- Gini: EBM {t.loc["EBM","Gini"]:.3f}, KS: EBM {t.loc["EBM","KS"]:.3f}
- Rejecting the riskiest 10% with EBM: bad rate {base:.2%} -> {r90["EBM"]:.2%} ({(1 - r90["EBM"]/base):.0%} fewer bad loans per approved loan)
- Top risk drivers: {", ".join(imp.sort_values(ascending=False).index[:5])}
''')